In [1]:
import pandas as pd
import numpy as np

usage = pd.read_csv("../data/processed/usage_shares.csv")
emp = pd.read_csv("../data/processed/employment_shares.csv")

codes = {"AUT": "AT", "BEL": "BE", "CZE": "CZ", "DNK": "DK", "FIN": "FI", "FRA": "FR",
         "DEU": "DE", "GRC": "EL", "IRL": "IE", "ITA": "IT", "NLD": "NL", "POL": "PL",
         "PRT": "PT", "ROU": "RO", "ESP": "ES", "SWE": "SE",
         "BGR": "BG", "HUN": "HU", "LTU": "LT", "SVK": "SK"}
usage = usage[usage["geo_id"].isin(codes)].copy()
usage["geo"] = usage["geo_id"].map(codes)


s = usage.pivot_table(index=["geo", "isco2"], columns="date_start", values="pct_isco").reset_index()
s = s.rename(columns={"2026-04-01": "s_apr", "2026-05-01": "s_may"})
s[["s_apr", "s_may"]] = s[["s_apr", "s_may"]] / 100
e = emp.pivot_table(index=["geo", "isco08"], columns="year", values="share").reset_index()
e = e.rename(columns={"isco08": "isco2", 2015: "e_2015", 2024: "e_2024", 2025: "e_2025"})


panel = s.merge(e, on=["geo", "isco2"], how="inner")
main16 = ["AT", "BE", "CZ", "DK", "FI", "FR", "DE", "EL", "IE", "IT", "NL", "PL", "PT", "RO", "ES", "SE"]
panel["main_sample"] = panel["geo"].isin(main16)

for col in ["s_may", "s_apr", "e_2025", "e_2024", "e_2015"]:
    panel["ln_" + col] = np.log(panel[col])

In [2]:
print(panel.shape, panel["main_sample"].sum())
print("Duplicates:", panel.duplicated(["geo", "isco2"]).sum())
print(panel[panel["main_sample"]].groupby("geo").size())

(578, 13) 484
Duplicates: 0
geo
AT    28
BE    31
CZ    28
DE    35
DK    27
EL    27
ES    35
FI    27
FR    36
IE    26
IT    33
NL    32
PL    31
PT    29
RO    28
SE    31
dtype: int64


In [3]:
panel.to_csv("../data/processed/panel.csv", index=False)